# 06 - Location Prediction Models

**CASEFILE: AI-Powered Missing Person Investigation System**  
*Phase 6: Multi-Model Supervised Location Prediction & Search Quadrant Ranking*

---

### Overview
When an individual is reported missing, search-and-rescue teams face the critical challenge of identifying which geographical sector should receive immediate search priority. 

This notebook demonstrates:
1. **Four-Model Architecture Benchmark**: Comparative evaluation of four classification algorithms:
   - **Random Forest Classifier**
   - **XGBoost Classifier**
   - **Gradient Boosting Classifier**
   - **K-Nearest Neighbors (KNN)**
2. **Operational Top-K Accuracy**: Evaluating model utility across Top-1, Top-3, and Top-5 candidate sectors.
3. **Model Selection**: Gradient Boosting Classifier selected as the primary estimator (50.0% Top-1, 100.0% Top-5 coverage).
4. **Feature Importance & Interpretability**: Identifying key drivers governing destination probabilities.
5. **Confusion Matrix Analysis**: Evaluating inter-area classification boundaries.

> **DISCLAIMER & ETHICAL USAGE STATEMENT**  
> All case records and person profiles are synthetic simulations generated for academic research purposes. Machine learning location predictions provide probabilistic sector rankings to support operational resource allocation. They must never replace standard police procedures, emergency alerts, or eyewitness reports.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Setup visualization styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Directory paths
DATA_DIR = '../data' if os.path.exists('../data') else 'data'
REPORTS_DIR = '../reports' if os.path.exists('../reports') else 'reports'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'

print("Location prediction environment initialized successfully.")
print(f"Data Directory: {DATA_DIR}")
print(f"Models Directory: {MODELS_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

## 1. Load Synthetic Cases Dataset

We load `data/synthetic/cases.csv`, which documents missing person investigation cases with situational context, demographic profiles, last known coordinates, mobility history, and target destination areas.

In [ ]:
import sys
sys.path.insert(0, '..')

cases_path = os.path.join(DATA_DIR, 'synthetic', 'cases.csv')
cases_df = pd.read_csv(cases_path)

print(f"Total synthetic case profiles: {len(cases_df)}")
print(f"Features ({len(cases_df.columns)}): {cases_df.columns.tolist()}")
display(cases_df[['Case_ID', 'Age_Group', 'Gender', 'Day', 'Weather', 
                  'Last_Latitude', 'Last_Longitude', 'Time_Since_Last_Seen', 'Target_Area']].head(5))

## 2. Four-Model Performance Benchmark & Evaluation

Four candidate classification models were trained and evaluated on an 80/20 train/test split:
1. **Random Forest Classifier**: Ensemble of 200 bagged decision trees with maximum depth 15.
2. **XGBoost Classifier**: 200 boosted trees with learning rate $\eta = 0.1$ and multi-class log-loss.
3. **Gradient Boosting Classifier**: 150 additive decision trees with maximum depth 6.
4. **K-Nearest Neighbors (KNN)**: Distance-weighted neighborhood retrieval with $k = 7$.

In operational search-and-rescue, standard Top-1 Accuracy is complemented by **Top-K Accuracy**:
- **Top-1 Accuracy**: Exact predicted sector matches the true location.
- **Top-3 Accuracy**: True location is within the top 3 recommended search sectors.
- **Top-5 Accuracy**: True location is within the top 5 recommended search sectors.

In [ ]:
import sys
sys.path.insert(0, '..')

# Load model comparison table
comp_path = os.path.join(REPORTS_DIR, 'model_comparison.csv')
comp_df = pd.read_csv(comp_path)

# Format nicely for presentation
formatted_comp = comp_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1', 'Top-1 Acc', 'Top-3 Acc', 'Top-5 Acc']:
    if col in formatted_comp.columns:
        formatted_comp[col] = formatted_comp[col].apply(lambda v: f"{v * 100:.1f}%")

print("=" * 75)
print("               FOUR-MODEL PERFORMANCE COMPARISON TABLE")
print("=" * 75)
display(formatted_comp)

In [ ]:
import sys
sys.path.insert(0, '..')

# Visualize Top-K search accuracy across models
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(comp_df))
bar_width = 0.25

rects1 = ax.bar(x - bar_width, comp_df['Top-1 Acc'] * 100, bar_width, 
                label='Top-1 Accuracy', color='#3498db', edgecolor='black', alpha=0.85)
rects2 = ax.bar(x, comp_df['Top-3 Acc'] * 100, bar_width, 
                label='Top-3 Accuracy', color='#2ecc71', edgecolor='black', alpha=0.85)
rects3 = ax.bar(x + bar_width, comp_df['Top-5 Acc'] * 100, bar_width, 
                label='Top-5 Accuracy', color='#e67e22', edgecolor='black', alpha=0.85)

ax.set_title("Model Accuracy Across Operational Search Radii (Top-K)", fontsize=13, fontweight='bold')
ax.set_ylabel("Accuracy (%)")
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Model'], fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='upper left', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Add direct data labels
for i, model in enumerate(comp_df['Model']):
    ax.text(i - bar_width, comp_df.loc[i, 'Top-1 Acc'] * 100 + 2, 
            f"{comp_df.loc[i, 'Top-1 Acc']*100:.0f}%", ha='center', fontsize=9, fontweight='bold')
    ax.text(i, comp_df.loc[i, 'Top-3 Acc'] * 100 + 2, 
            f"{comp_df.loc[i, 'Top-3 Acc']*100:.0f}%", ha='center', fontsize=9, fontweight='bold')
    ax.text(i + bar_width, comp_df.loc[i, 'Top-5 Acc'] * 100 + 2, 
            f"{comp_df.loc[i, 'Top-5 Acc']*100:.0f}%", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Best Model Selection: Gradient Boosting Classifier

### Selection Rationale:
- **Top-1 Accuracy**: **50.0%** (ties Random Forest).
- **Precision (Weighted)**: **52.0%** (highest among all evaluated models).
- **F1-Score (Weighted)**: **47.6%** (highest comprehensive balance).
- **Top-5 Accuracy**: **100.0%** — In 100% of test cases, the missing person's true location was successfully captured within the top-5 recommended search sectors.

We load the production model artifact from `models/location_model.pkl`.

In [ ]:
import sys
sys.path.insert(0, '..')

model_path = os.path.join(MODELS_DIR, 'location_model.pkl')
best_model = joblib.load(model_path)

print("=" * 50)
print("      PRODUCTION MODEL SPECIFICATION")
print("=" * 50)
print(f"Model Architecture:    {type(best_model).__name__}")
print(f"Number of Estimators:  {best_model.n_estimators}")
print(f"Max Tree Depth:        {best_model.max_depth}")
print(f"Learning Rate:         {best_model.learning_rate}")
print(f"Target Classes (Areas): {len(best_model.classes_)}")
print("=" * 50)

## 4. Feature Importance & Model Interpretability

Understanding why the model favors certain areas is essential for tactical transparency.

Key feature weights from the pipeline run:
- `Last_Longitude` (**28.7%**) & `Last_Latitude` (**24.0%**): Spatial proximity to the point of disappearance governs over **52%** of the prediction weight.
- `Day` (**9.8%**): Day of the week captures shifts between habitual weekday routes and weekend destinations.
- `Time_Since_Last_Seen` (**9.5%**): Elapsed time dictates the expanding mobility radius.
- `hour` (**8.0%**): Diurnal travel patterns (daytime transit vs nighttime dwelling).

In [ ]:
import sys
sys.path.insert(0, '..')

# Display feature importance report image
fi_img = os.path.join(REPORTS_DIR, 'feature_importance_GradientBoosting.png')
if os.path.exists(fi_img):
    print("Loading reports/feature_importance_GradientBoosting.png:")
    display(Image(filename=fi_img, width=750))
else:
    print(f"Warning: {fi_img} not found.")

## 5. Model Evaluation: Confusion Matrix

The confusion matrix evaluates pairwise classification fidelity across all geographic destination sectors, highlighting adjacent areas with overlapping feature boundaries.

In [ ]:
import sys
sys.path.insert(0, '..')

# Display confusion matrix report image
cm_img = os.path.join(REPORTS_DIR, 'confusion_matrix_GradientBoosting.png')
if os.path.exists(cm_img):
    print("Loading reports/confusion_matrix_GradientBoosting.png:")
    display(Image(filename=cm_img, width=650))
else:
    print(f"Warning: {cm_img} not found.")

## 6. End-to-End Real-Time Inference Simulation

We demonstrate an active investigation query using the trained pipeline. Given last seen coordinates and incident context, the model outputs ranked search sectors with calibrated probabilities and priority tiers.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.prediction import predict_case_location

# Load label encoders
encoders_path = os.path.join(MODELS_DIR, 'prediction_encoders.pkl')
encoders = joblib.load(encoders_path) if os.path.exists(encoders_path) else {}

# Simulated missing person incident scenario:
# Elderly woman last seen in eastern transit corridor on a rainy Friday evening
incident_case = {
    'Age_Group': '65+',
    'Gender': 'Female',
    'Day': 'Friday',
    'Weather': 'Rain',
    'hour': 18,
    'Last_Latitude': 39.920,
    'Last_Longitude': 116.293,
    'Average_Distance': 65.0,
    'Average_Speed': 2.8,
    'Time_Since_Last_Seen': 24
}

predictions = predict_case_location(best_model, incident_case, encoders, top_n=3)

print("=" * 65)
print("     TACTICAL SEARCH SECTOR PRIORITIZATION RECOMMENDATIONS")
print("=" * 65)
display(pd.DataFrame(predictions))

## 7. Conclusions & Investigative Handoff

- **Selected Architecture**: Gradient Boosting outperformed Random Forest, XGBoost, and KNN on precision (52.0%) and F1 (47.6%), while delivering **100% Top-5 coverage**.
- **Decision Transparency**: Feature importance confirms spatial anchors (`Last_Longitude`, `Last_Latitude`) account for 52% of model confidence, validated by time elapsed and temporal factors.
- **Downstream Integration**: Predicted sector probabilities feed directly into the **Search Priority Index** and **Route Prediction** modules.